In [0]:
from lib.ingestors import IngestorIncremental, Ingestor    
import delta

In [0]:
input_format = dbutils.widgets.get('input_format')
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
table_name = dbutils.widgets.get('table_name')
load_type = dbutils.widgets.get('load_type')

In [0]:
if load_type == 'incremental':
    ingest = IngestorIncremental(
        catalog=catalog, 
        schema=schema, 
        table_name=table_name,    
        spark = spark,
        input_format=input_format
)
else:
    ingest = Ingestor(
        catalog=catalog, 
        schema=schema, 
        table_name=table_name,
        spark = spark,
        input_format=input_format)

In [0]:
df = ingest.load()


In [0]:
query = """
SELECT
  CAST(currency AS string) as currency
  , CAST(marketCap AS DOUBLE) as marketCap
  , CAST(shortName AS string) as shortName
  , CAST(longName AS string) as longName
  , CAST(regularMarketChange AS DOUBLE) as regularMarketChange
  , CAST(regularMarketChangePercent AS DOUBLE) as regularMarketChangePercent
  , CAST(regularMarketTime AS timestamp) as regularMarketTime
  , CAST(regularMarketPrice AS DOUBLE) as regularMarketPrice
  , CAST(regularMarketDayHigh AS DOUBLE) as regularMarketDayHigh
  , CAST(regularMarketDayRange AS string) as regularMarketDayRange
  , CAST(regularMarketDayLow AS DOUBLE) as regularMarketDayLow
  , CAST(regularMarketVolume AS BIGINT) as regularMarketVolume
  , CAST(regularMarketPreviousClose AS DOUBLE) as regularMarketPreviousClose
  , CAST(regularMarketOpen AS DOUBLE) as regularMarketOpen
  , CAST(fiftyTwoWeekRange AS string) as fiftyTwoWeekRange
  , CAST(fiftyTwoWeekLow AS DOUBLE) as fiftyTwoWeekLow
  , CAST(fiftyTwoWeekHigh AS DOUBLE) as fiftyTwoWeekHigh
  , CAST(symbol AS string) as symbol
  , CAST(logourl AS string) as logourl
  , historicalDataPrice 
  , summaryProfile
  , CAST(priceEarnings AS double) AS priceEarnings
  , CAST(earningsPerShare AS DOUBLE) AS earningsPerShare
  , CAST(loaded_at AS timestamp) AS loaded_at

FROM view_tickers
QUALIFY row_number() over (partition by symbol order by regularMarketTime desc) = 1"""

df1 = spark.sql(query)
df1.display()



In [0]:
df.write.format('delta').mode('overwrite').option("overwriteSchema", "true").saveAsTable('bronze.brapi.tests')
x = delta.DeltaTable.forName(spark, 'bronze.brapi.tests')
merge = (
    x.alias("t").merge(
        df1.alias("s"),
        "t.symbol = s.symbol and t.regularMarketTime = s.regularMarketTime")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

result = spark.sql("select * from bronze.brapi.tests")
result.display()